# Data Quality Assessment and Remediation
Credit Application Dataset

## Structure
1. Dataset Overview
2. Completeness Assessment
3. Duplicate Record Analysis
4. Data Type Inconsistencies
5. Inconsistent Categorical Encoding
6. Invalid / Implausible Values
7. GDPR Compliance
8. Final Dataset Overview


In [1]:
# ============================================================
# Imports
# ============================================================

# Standard library
import json
import os
from pathlib import Path

# Data handling
import pandas as pd
import numpy as np

In [2]:
start = Path.cwd()

for p in [start] + list(start.parents):
    candidate = p / "data" / "data.json"
    if candidate.exists():
        DATA_FILE = candidate
        break
else:
    raise FileNotFoundError("Could not locate data/data.json in parent folders")

print("Using:", DATA_FILE)

with open(DATA_FILE, "r", encoding="utf-8") as f:
    raw = json.load(f)

df = pd.json_normalize(raw, sep=".")

print("Loaded records:", len(raw))
print("DataFrame shape:", df.shape)

Using: /Users/sivert/Desktop/dego-project-team2/data/data.json
Loaded records: 502
DataFrame shape: (502, 21)


## 1. Dataset Overview

This notebook performs a structured data quality assessment and remediation of the Credit Application dataset.

Each row represents a loan application.

### Identifier Clarification

The `_id` variable is assumed to represent a unique loan application identifier (not a customer identifier).  
Therefore:

- Duplicate `_id` values indicate duplicate application records.
- Each `_id` should appear only once in a clean dataset.

### Data Quality Framework

The following data quality dimensions are assessed:

- **Completeness** – Missing or incomplete values
- **Consistency** – Logical contradictions between fields
- **Validity** – Incorrect formats or invalid values
- **Accuracy / Plausibility** – Realistic business ranges

In [3]:
# --------------------------------------------------
# 1. Basic Dataset Overview
# --------------------------------------------------

# The raw JSON file contains a list of loan applications.
# We normalize nested structures to create a flat analytical table.
df = pd.json_normalize(raw, sep=".")

# Inspect basic structural properties of the dataset

print("Dataset shape (rows, columns):", df.shape)

print("\nTotal number of columns:", len(df.columns))

print("\nColumn names:")
for col in df.columns:
    print("-", col)

Dataset shape (rows, columns): (502, 21)

Total number of columns: 21

Column names:
- _id
- spending_behavior
- processing_timestamp
- applicant_info.full_name
- applicant_info.email
- applicant_info.ssn
- applicant_info.ip_address
- applicant_info.gender
- applicant_info.date_of_birth
- applicant_info.zip_code
- financials.annual_income
- financials.credit_history_months
- financials.debt_to_income
- financials.savings_balance
- decision.loan_approved
- decision.rejection_reason
- loan_purpose
- decision.interest_rate
- decision.approved_amount
- financials.annual_salary
- notes


In [4]:
# Inspect data types and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 502 entries, 0 to 501
Data columns (total 21 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   _id                               502 non-null    object 
 1   spending_behavior                 502 non-null    object 
 2   processing_timestamp              62 non-null     object 
 3   applicant_info.full_name          502 non-null    object 
 4   applicant_info.email              502 non-null    object 
 5   applicant_info.ssn                497 non-null    object 
 6   applicant_info.ip_address         497 non-null    object 
 7   applicant_info.gender             501 non-null    object 
 8   applicant_info.date_of_birth      501 non-null    object 
 9   applicant_info.zip_code           501 non-null    object 
 10  financials.annual_income          497 non-null    object 
 11  financials.credit_history_months  502 non-null    int64  
 12  financia

## 2. Completeness Assessment


In [5]:
# ============================================================
# 2. Completeness Assessment
# ============================================================

# ------------------------------------------------------------
# Calculate completeness metrics per column
# ------------------------------------------------------------

completeness_summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null_count": df.notna().sum(),
    "missing_count": df.isna().sum(),
})

# Percentage missing
completeness_summary["missing_pct"] = (
    completeness_summary["missing_count"] / len(df) * 100
).round(2)

# Sort by most missing values first
completeness_summary = completeness_summary.sort_values(
    "missing_pct", ascending=False
)

print("Completeness summary (sorted by missing %):")
completeness_summary

Completeness summary (sorted by missing %):


,dtype,non_null_count,missing_count,missing_pct
notes,object,2,500,99.60
financials.annual_salary,float64,5,497,99.00
loan_purpose,object,50,452,90.04
processing_timestamp,object,62,440,87.65
decision.rejection_reason,object,210,292,58.17
decision.approved_amount,float64,292,210,41.83
decision.interest_rate,float64,292,210,41.83
financials.annual_income,object,497,5,1.00
applicant_info.ip_address,object,497,5,1.00
applicant_info.ssn,object,497,5,1.00


## Missing Data Overview

The completeness assessment reveals substantial variation in data availability across variables.

### Nearly completely missing (>85%)

The following variables contain very limited information:

- `notes` (99.6%)
- `financials.annual_salary` (99.0%)
- `loan_purpose` (90.0%)
- `processing_timestamp` (87.7%)

These fields have extremely high missingness and may provide limited analytical value without strong justification or external enrichment.

---

### Moderately missing (40–60%)

The following decision-related variables show moderate missingness:

- `decision.rejection_reason` (58.2%)
- `decision.approved_amount` (41.8%)
- `decision.interest_rate` (41.8%)

This pattern suggests that missingness may be **structurally driven**, likely depending on loan approval status.  
This will be formally validated in the Consistency section.

---

### Low missing (<5%)

Minor missingness is observed in:

- `financials.annual_income` (1.0%)
- `applicant_info.ssn` (1.0%)
- `applicant_info.ip_address` (1.0%)
- `applicant_info.date_of_birth` (0.2%)
- `applicant_info.zip_code` (0.2%)
- `applicant_info.gender` (0.2%)

These levels are generally manageable and may be addressed through controlled imputation or exclusion strategies.

---

Overall, completeness issues are concentrated in decision-related and auxiliary fields, while core financial and behavioral variables remain largely complete.

In [6]:
# ------------------------------------------------------------
# Completeness check: Is missingness structurally driven by approval status?
# We compare missing rates for key decision fields between approved vs not approved.
# ------------------------------------------------------------

cols_to_check = [
    "decision.interest_rate",
    "decision.approved_amount",
    "decision.rejection_reason",
    "processing_timestamp",
]

# Missing rate (in %) within each approval group
missing_by_approval = (
    df.groupby("decision.loan_approved")[cols_to_check]
      .apply(lambda g: g.isna().mean() * 100)
      .round(2)
)

print("Missing rate (%) by approval status:")
missing_by_approval

Missing rate (%) by approval status:


,decision.interest_rate,decision.approved_amount,decision.rejection_reason,processing_timestamp
decision.loan_approved,,,,
False,100.0,100.0,0.0,89.52
True,0.0,0.0,100.0,86.30


## Structural Missingness Validation

The missingness analysis by approval status confirms that several decision-related fields are **structurally missing**.

### Key observations:

- `decision.interest_rate` and `decision.approved_amount`  
  - 100% missing for rejected loans  
  - 0% missing for approved loans  

- `decision.rejection_reason`  
  - 0% missing for rejected loans  
  - 100% missing for approved loans  

This clearly indicates that missingness is driven by business logic rather than data quality errors.

In other words:
- Interest rate and approved amount are only applicable for approved loans.
- Rejection reason is only applicable for rejected loans.

Therefore, these variables should **not be treated as data quality defects**, but as structurally conditional attributes.

---

`processing_timestamp`, however, remains highly missing across both groups (~86–89%), suggesting a potential logging or system-level issue rather than structural logic.

## 3. Duplicate Record Analysis

In [7]:
# ============================================================
# 3. Duplicate Record Analysis
# ============================================================

# ------------------------------------------------------------
# Check for duplicate application IDs
# ------------------------------------------------------------

duplicate_mask = df.duplicated(subset="_id", keep=False)
duplicates = df[duplicate_mask].sort_values("_id")

print("Number of duplicate rows:", duplicates.shape[0])
print("Number of duplicated application IDs:", duplicates["_id"].nunique())

duplicates

Number of duplicate rows: 4
Number of duplicated application IDs: 2


,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,...,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes
383,app_001,"[{'category': 'Fitness', 'amount': 576}]",NaN,Stephanie Nguyen,stephanie.nguyen47@mail.com,427-90-1892,10.121.120.213,Female,1986-05-27,90230,...,37,0.42,0,False,high_dti_ratio,NaN,NaN,NaN,NaN,NaN
455,app_001,"[{'category': 'Fitness', 'amount': 576}]",NaN,Stephanie Nguyen,stephanie.nguyen47@mail.com,NaN,NaN,NaN,NaN,NaN,...,37,0.42,0,False,high_dti_ratio,NaN,NaN,NaN,NaN,DUPLICATE_ENTRY_ERROR
8,app_042,"[{'category': 'Insurance', 'amount': 153}, {'c...",NaN,Joseph Lopez,joseph.lopez1@gmail.com,652-70-5530,192.168.91.142,Male,1990-05-04,10044,...,43,0.41,15974,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
354,app_042,"[{'category': 'Insurance', 'amount': 153}, {'c...",NaN,Joseph Lopez,joseph.lopez1@gmail.com,652-70-5530,192.168.91.142,Male,1990-05-04,10044,...,43,0.41,15974,False,algorithm_risk_score,NaN,NaN,NaN,NaN,RESUBMISSION


In [8]:
# ------------------------------------------------------------
# Check whether duplicate rows are fully identical
# ------------------------------------------------------------

# Convert list-like columns to string for reliable comparison
df_for_compare = df.copy()
df_for_compare["spending_behavior"] = df_for_compare["spending_behavior"].astype(str)

# Extract duplicates again (after conversion)
dups_compare = df_for_compare[df_for_compare.duplicated(subset="_id", keep=False)]

# Count number of unique rows per _id
duplicate_uniqueness = (
    dups_compare
    .groupby("_id")
    .nunique(dropna=False)
)

duplicate_uniqueness

,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes
_id,,,,,,,,,,,,,,,,,,,,
app_001,1,1,1,1,2,2,2,2,2,1,1,1,1,1,1,1,1,1,1,2
app_042,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,2


In [9]:
# ------------------------------------------------------------
# Resolve duplicates by keeping the most complete record
# ------------------------------------------------------------

# Count number of non-null values per row
df["_non_null_count"] = df.notna().sum(axis=1)

# Sort by completeness (descending)
df_sorted = df.sort_values("_non_null_count", ascending=False)

# Keep most complete record per _id
df_deduplicated = df_sorted.drop_duplicates(subset="_id", keep="first")

# Drop helper column
df_deduplicated = df_deduplicated.drop(columns="_non_null_count")

print("Original shape:", df.shape)
print("After deduplication:", df_deduplicated.shape)

Original shape: (502, 22)
After deduplication: (500, 21)


In [10]:
# From this point on, we use the deduplicated dataset for all analyses
df = df_deduplicated.copy()

## Duplicate Record Resolution

Two duplicated application identifiers were identified (`app_001` and `app_042`), resulting in 4 duplicate rows in total.

The duplicate entries were not fully identical. In particular, `app_001` contained conflicting personal information (SSN, IP address, gender, date of birth, zip code, and notes), indicating a **consistency violation** rather than a simple system duplication.

### Data Quality Dimension Mapping

- **Consistency**: Duplicate primary identifiers violate entity uniqueness constraints.
- **Accuracy**: Conflicting personal attributes create uncertainty about the correct record.
- **Completeness**: Duplicate resolution strategy preserved the most complete record.

### Remediation Strategy

Duplicates were resolved by retaining the record with the highest number of non-null fields per `_id`, ensuring maximum information retention while enforcing unique identifiers.

After remediation:
- Dataset reduced from 502 to 500 records.
- Each `_id` now uniquely identifies a loan application.

## 4. Data Type Inconsistencies

In [11]:
# ============================================================
# 4. Data Type Inconsistencies
# Example: financials.annual_income
# ============================================================

col = "financials.annual_income"

# ----------------------------
# 1) Type distribution BEFORE conversion
# ----------------------------

type_counts_before = df[col].dropna().apply(type).value_counts()

print("Type distribution BEFORE conversion:")
print(type_counts_before)

total_rows = len(df)

# Count string values properly
string_count = type_counts_before.get(type("example"), 0)
string_pct = (string_count / total_rows) * 100

missing_before = df[col].isna().sum()
missing_before_pct = (missing_before / total_rows) * 100



# ----------------------------
# 2) Remediation: convert to numeric
#    (invalid parsing -> NaN)
# ----------------------------

df[col] = pd.to_numeric(df[col], errors="coerce")


# ----------------------------
# 3) Verify AFTER conversion
# ----------------------------

type_counts_after = df[col].dropna().apply(type).value_counts()

missing_after = df[col].isna().sum()
missing_after_pct = (missing_after / total_rows) * 100

print("\nType distribution AFTER conversion:")
print(type_counts_after)


Type distribution BEFORE conversion:
financials.annual_income
<class 'int'>      486
<class 'str'>        8
<class 'float'>      1
Name: count, dtype: int64

Type distribution AFTER conversion:
financials.annual_income
<class 'float'>    495
Name: count, dtype: int64


In [12]:
# ============================================================
# Inspect rows where annual_income is missing
# ============================================================

col = "financials.annual_income"

# Filter missing rows
missing_rows = df[df[col].isna()]

print(f"Number of rows with missing {col}: {missing_rows.shape[0]}")
print("\nApplication IDs with missing income:")
print(missing_rows["_id"].values)

# Display full rows for inspection
missing_rows

Number of rows with missing financials.annual_income: 5

Application IDs with missing income:
['app_421' 'app_449' 'app_436' 'app_463' 'app_479']


,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,...,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes
94,app_421,"[{'category': 'Utilities', 'amount': 563}, {'c...",NaN,Donald Baker,donald.baker60@icloud.com,344-50-4287,192.168.147.249,Male,1982-07-17,10027,...,44,0.06,5018,True,NaN,NaN,6.0,45000.0,46000.0,NaN
149,app_449,"[{'category': 'Insurance', 'amount': 498}]",NaN,Lisa Roberts,lisa.roberts99@outlook.com,833-71-7935,172.23.144.234,Female,1968-11-09,90234,...,86,0.07,27467,True,NaN,NaN,5.7,54000.0,75000.0,NaN
76,app_436,"[{'category': 'Insurance', 'amount': 339}]",NaN,Amanda Torres,amanda.torres59@gmail.com,135-51-1195,172.18.137.22,F,1998-06-02,90217,...,35,0.13,10311,True,NaN,NaN,3.9,63000.0,45000.0,NaN
141,app_463,"[{'category': 'Travel', 'amount': 690}, {'cate...",NaN,Emma Clark,emma.clark61@outlook.com,976-47-3536,10.194.129.196,Female,1975-04-26,90205,...,50,0.29,66878,False,algorithm_risk_score,NaN,NaN,NaN,86000.0,NaN
99,app_479,"[{'category': 'Transportation', 'amount': 136}]",NaN,Sandra Jones,sandra.jones59@outlook.com,424-34-1670,172.25.239.70,F,1983/11/08,90205,...,56,0.11,15630,False,algorithm_risk_score,NaN,NaN,NaN,94000.0,NaN


In [13]:
# ============================================================
# Check relationship between annual_income and annual_salary
# ============================================================

income_col = "financials.annual_income"
salary_col = "financials.annual_salary"

# 1) Rows where salary is NOT missing
salary_present = df[df[salary_col].notna()]

print(f"Number of rows with non-missing annual_salary: {salary_present.shape[0]}")
print("Application IDs with salary present:")
print(salary_present["_id"].values)

# 2) Check if these same rows are the ones missing income
salary_and_missing_income = salary_present[salary_present[income_col].isna()]

print("\nRows with salary present AND income missing:")
print(salary_and_missing_income["_id"].values)

# 3) Check if any row has BOTH income and salary present
both_present = df[
    df[income_col].notna() &
    df[salary_col].notna()
]

print(f"\nRows with BOTH income and salary present: {both_present.shape[0]}")
print(both_present["_id"].values)

Number of rows with non-missing annual_salary: 5
Application IDs with salary present:
['app_421' 'app_449' 'app_436' 'app_463' 'app_479']

Rows with salary present AND income missing:
['app_421' 'app_449' 'app_436' 'app_463' 'app_479']

Rows with BOTH income and salary present: 0
[]


In [14]:
# ============================================================
# Resolve redundancy between annual_income and annual_salary
# ============================================================

income_col = "financials.annual_income"
salary_col = "financials.annual_salary"

# Create unified income column
df["financials.total_income"] = df[income_col].combine_first(df[salary_col])

# Check missing after merge
missing_total_income = df["financials.total_income"].isna().sum()
print("Missing values in unified income column:", missing_total_income)

# Drop redundant columns
df = df.drop(columns=[income_col, salary_col])

print("Columns after cleanup:")
print(df.columns)

Missing values in unified income column: 0
Columns after cleanup:
Index(['_id', 'spending_behavior', 'processing_timestamp',
       'applicant_info.full_name', 'applicant_info.email',
       'applicant_info.ssn', 'applicant_info.ip_address',
       'applicant_info.gender', 'applicant_info.date_of_birth',
       'applicant_info.zip_code', 'financials.credit_history_months',
       'financials.debt_to_income', 'financials.savings_balance',
       'decision.loan_approved', 'decision.rejection_reason', 'loan_purpose',
       'decision.interest_rate', 'decision.approved_amount', 'notes',
       'financials.total_income'],
      dtype='object')


### Redundant Financial Encoding – Income vs Salary

The dataset contained two variables:

- `financials.annual_income`
- `financials.annual_salary`

Analysis revealed:

- 5 records contained salary but no income.
- The same 5 records had missing income.
- No records contained both variables simultaneously.

This indicates a **redundant representation of the same financial concept**, rather than two distinct variables.

#### Data Quality Dimensions

- **Consistency**: Two columns representing the same concept.
- **Completeness**: Income values were fragmented across two variables.
- **Accuracy**: Separate storage could distort financial analysis.

#### Remediation

The variables were merged into a unified column


In [ ]:
# ============================================================
# Type audit: identify "object" columns that should be numeric
# ============================================================

object_cols = df.select_dtypes(include=["object"]).columns.tolist()

# Columns that are expected to be numeric (based on schema / common sense)
expected_numeric = [
    "financials.total_income",          
    "financials.savings_balance",
    "financials.credit_history_months",
    "financials.debt_to_income",
    "decision.interest_rate",
    "decision.approved_amount"
]

# Keep only those that actually exist in df
expected_numeric = [c for c in expected_numeric if c in df.columns]

suspect_numeric = [c for c in expected_numeric if c in object_cols]

print("Object columns:", object_cols)
print("\nExpected numeric columns present:", expected_numeric)
print("\nSuspect numeric columns stored as object:", suspect_numeric)

# Show a few sample values to understand what's inside
for c in suspect_numeric:
    print(f"\n--- Sample values in {c} ---")
    print(df[c].dropna().astype(str).head(10).tolist())

Object columns: ['_id', 'spending_behavior', 'processing_timestamp', 'applicant_info.full_name', 'applicant_info.email', 'applicant_info.ssn', 'applicant_info.ip_address', 'applicant_info.gender', 'applicant_info.date_of_birth', 'applicant_info.zip_code', 'decision.rejection_reason', 'loan_purpose', 'notes']

Expected numeric columns present: ['financials.total_income', 'financials.savings_balance', 'financials.credit_history_months', 'financials.debt_to_income', 'decision.interest_rate', 'decision.approved_amount']

Suspect numeric columns stored as object: []


In [18]:
# ============================================================
# Standardize date_of_birth -> datetime (robust parsing)
# ============================================================

dob_col = "applicant_info.date_of_birth"

# Keep original for audit (optional but nice for reporting)
df["_dob_raw"] = df[dob_col]

# Robust parsing: try normal parsing, then dayfirst fallback
dob_parsed = pd.to_datetime(df[dob_col], errors="coerce", infer_datetime_format=True)

# If some failed, try dayfirst=True for those rows
failed_mask = dob_parsed.isna() & df[dob_col].notna()
dob_parsed_dayfirst = pd.to_datetime(df.loc[failed_mask, dob_col], errors="coerce", dayfirst=True)

dob_parsed.loc[failed_mask] = dob_parsed_dayfirst

df[dob_col] = dob_parsed

print("Total rows:", len(df))
print("DOB missing (original):", df["_dob_raw"].isna().sum())
print("DOB unparseable -> NaT:", df[dob_col].isna().sum() - df["_dob_raw"].isna().sum())

# Show a few examples of values that couldn't be parsed
bad_examples = df.loc[df[dob_col].isna() & df["_dob_raw"].notna(), ["_id", "_dob_raw"]].head(10)
print("\nExamples of unparseable DOB values:")
display(bad_examples)


Total rows: 500
DOB missing (original): 0
DOB unparseable -> NaT: 96

Examples of unparseable DOB values:


/var/folders/tx/4lmfklxn1v1b390w_hlxcktc0000gn/T/ipykernel_88222/1934548303.py:11: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  dob_parsed = pd.to_datetime(df[dob_col], errors="coerce", infer_datetime_format=True)
/var/folders/tx/4lmfklxn1v1b390w_hlxcktc0000gn/T/ipykernel_88222/1934548303.py:15: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dob_parsed_dayfirst = pd.to_datetime(df.loc[failed_mask, dob_col], errors="coerce", dayfirst=True)


,_id,_dob_raw
113,app_223,2000/12/28
373,app_244,17/03/1996
410,app_209,1988/03/16
60,app_228,14/12/1987
59,app_160,29/12/1982
277,app_157,30/12/1982
287,app_153,1979/01/24
165,app_226,1982/04/18
324,app_235,1997/08/31
37,app_178,20/07/1997


In [19]:
# ============================================================
# DOB remediation: normalize separators + multi-format parsing
# ============================================================

dob_col = "applicant_info.date_of_birth"

# Normalize separators ( / -> - ) and strip spaces
dob_clean = df["_dob_raw"].astype(str).str.strip().str.replace("/", "-", regex=False)

def parse_multi_format(series: pd.Series) -> pd.Series:
    s = series.copy()
    out = pd.Series(pd.NaT, index=s.index)

    # Try formats in order (most unambiguous first)
    formats = ["%Y-%m-%d", "%d-%m-%Y", "%m-%d-%Y"]

    remaining = s.index
    for fmt in formats:
        parsed = pd.to_datetime(s.loc[remaining], format=fmt, errors="coerce")
        ok = parsed.notna()
        out.loc[remaining[ok]] = parsed.loc[ok]
        remaining = remaining[~ok]
        if len(remaining) == 0:
            break

    return out

df[dob_col] = parse_multi_format(dob_clean)

print("Total rows:", len(df))
print("Unparseable after multi-format parsing:", df[dob_col].isna().sum())

# Show examples that still fail
bad_examples = df.loc[df[dob_col].isna(), ["_id", "_dob_raw"]].head(15)
print("\nExamples still unparseable:")
display(bad_examples)

Total rows: 500
Unparseable after multi-format parsing: 4

Examples still unparseable:


,_id,_dob_raw
448,app_350,
462,app_165,
26,app_075,
275,app_120,


In [21]:
# ============================================================
# Date of Birth – Storage Overview
# ============================================================

dob_col = "applicant_info.date_of_birth"

print("Data type:")
print(df[dob_col].dtype)

print("\nNumber of non-null values:", df[dob_col].notna().sum())
print("Number of missing values:", df[dob_col].isna().sum())

print("\nMin date:", df[dob_col].min())
print("Max date:", df[dob_col].max())

print("\nSample stored values:")
display(df[dob_col].dropna().head(10))

Data type:
datetime64[ns]

Number of non-null values: 496
Number of missing values: 4

Min date: 1958-09-21 00:00:00
Max date: 2002-04-24 00:00:00

Sample stored values:


501   1981-06-03
123   1996-09-14
357   1982-11-17
113   2000-12-28
366   1991-04-23
369   1970-04-26
106   1991-12-11
373   1996-03-17
376   2001-07-27
100   1978-11-10
Name: applicant_info.date_of_birth, dtype: datetime64[ns]

## Date of Birth – Storage Overview

After remediation, the `applicant_info.date_of_birth` variable is now stored as a standardized datetime field.

### Current Storage Format

- **Data type:** `datetime64[ns]`
- **Format representation:** ISO standard (`YYYY-MM-DD`)
- **Missing values:** 4 records
- **Successfully parsed values:** 496 records

All previously inconsistent formats (e.g., `YYYY/MM/DD`, `DD/MM/YYYY`, mixed separators) have been normalized and converted into a consistent datetime representation.

---

### Data Quality Improvements

- **Consistency:** Multiple date formats were standardized into a single format.
- **Validity:** Dates are now stored as true datetime objects rather than free-text strings.
- **Accuracy:** Ambiguity between US and EU date formats has been eliminated through explicit parsing.
- **Completeness:** Only 4 records contain genuinely missing date values.

---

### Analytical Implication

Storing dates in a standardized datetime format ensures:

- Reliable age derivation  
- Accurate demographic segmentation  
- Reduced risk of parsing errors in downstream analysis  
- Improved compliance preparation under GDPR  

## 5. Inconsistent Categorical Encoding

In [ ]:
# ============================================================
# 5.1 Categorical encoding check: applicant_info.gender
# ============================================================

col = "applicant_info.gender"

print("Unique gender values (raw):")
display(df[col].value_counts(dropna=False))

print("\nRaw examples:")
display(df[[col]].dropna().sample(min(10, df[col].dropna().shape[0]), random_state=42))

Unique gender values (raw):


applicant_info.gender
Male      194
Female    193
F          58
M          53
            2
Name: count, dtype: int64


Raw examples:


,applicant_info.gender
68,Male
298,Female
231,Male
396,M
243,Male
33,Female
487,Male
326,M
294,Female
136,Female


In [24]:
# ============================================================
# Standardizing gender encoding
# ============================================================

col = "applicant_info.gender"

# Normalize case and strip spaces
df[col] = df[col].astype(str).str.strip().str.lower()

# Map variations to unified categories
gender_mapping = {
    "male": "male",
    "m": "male",
    "female": "female",
    "f": "female"
}

df[col] = df[col].map(gender_mapping)

# Replace unmapped (including former missing) with 'unknown'
df[col] = df[col].fillna("unknown")

print("Gender distribution AFTER standardization:")
display(df[col].value_counts(dropna=False))


Gender distribution AFTER standardization:


applicant_info.gender
female     251
male       247
unknown      2
Name: count, dtype: int64

## Categorical Encoding Standardization – Gender

### Issue Identified

The `applicant_info.gender` variable contained inconsistent categorical representations:

- "Male" and "M"
- "Female" and "F"
- Mixed capitalization
- Missing values

This represents an **inconsistent categorical encoding** issue, where multiple representations refer to the same logical category.

---

### Remediation Approach

The following transformations were applied:

1. Converted all values to lowercase
2. Trimmed whitespace
3. Mapped shorthand values:
   - `"m"` → `"male"`
   - `"f"` → `"female"`
4. Replaced unmapped or missing values with `"unknown"`

---

### Resulting Distribution

- **female:** 251  
- **male:** 247  
- **unknown:** 2  

---

### Data Quality Dimension Mapping

- **Consistency:** All gender values now follow a single standardized representation.
- **Validity:** Eliminated ambiguous shorthand encodings.
- **Completeness:** Explicit `"unknown"` category preserves missing information transparently.

This ensures reliable demographic analysis and prevents category fragmentation in downstream modeling.

In [25]:
# ============================================================
# 5.2 Categorical encoding check: loan_purpose
# ============================================================

col = "loan_purpose"

print("Loan purpose distribution (raw):")
display(df[col].value_counts(dropna=False))

print("\nUnique values:")
display(sorted(df[col].dropna().unique()))


Loan purpose distribution (raw):


loan_purpose
NaN                   450
medical                 8
education               7
wedding                 6
vacation                6
debt_consolidation      6
moving                  5
personal                4
auto                    3
home_improvement        3
business                2
Name: count, dtype: int64


Unique values:


['auto',
 'business',
 'debt_consolidation',
 'education',
 'home_improvement',
 'medical',
 'moving',
 'personal',
 'vacation',
 'wedding']

In [ ]:
# ============================================================
# Handle missing loan_purpose explicitly
# ============================================================

col = "loan_purpose"

df[col] = df[col].fillna("unknown")

print("Loan purpose distribution AFTER handling missing:")
display(df[col].value_counts())

Loan purpose distribution AFTER handling missing:


loan_purpose
unknown               450
medical                 8
education               7
wedding                 6
vacation                6
debt_consolidation      6
moving                  5
personal                4
auto                    3
home_improvement        3
business                2
Name: count, dtype: int64

## Categorical Encoding – Loan Purpose

### Issue Identified

The `loan_purpose` variable exhibited a very high level of missingness:

- 450 out of 500 records (90%) were missing
- Remaining categories were already consistently formatted (lowercase, snake_case)

Unlike `gender`, no encoding inconsistencies were detected.  
The primary issue was **missing data**, not categorical fragmentation.

---

### Remediation Approach

To ensure analytical transparency and consistent grouping behavior:

- Missing values were replaced with the explicit category `"unknown"`

This prevents silent exclusion during groupby operations and downstream analysis.

---

### Resulting Distribution

- **unknown:** 450  
- Remaining categories: medical, education, wedding, vacation, debt_consolidation, moving, personal, auto, home_improvement, business

---

### Data Quality Dimension Mapping

- **Completeness:** High missingness remains (90%), reducing analytical usefulness.
- **Consistency:** All categories are now explicitly defined.
- **Transparency:** Missing data is represented as `"unknown"` rather than implicit `NaN`.

Given the extremely high missing rate, this variable should be used cautiously in analytical or modeling contexts.

In [27]:
# ============================================================
# 5.3 Quick check: decision.rejection_reason
# ============================================================

col = "decision.rejection_reason"

print("Rejection reason distribution:")
display(df[col].value_counts(dropna=False))

Rejection reason distribution:


decision.rejection_reason
NaN                            292
algorithm_risk_score           169
insufficient_credit_history     23
high_dti_ratio                  12
low_income                       4
Name: count, dtype: int64

## Categorical Encoding – Rejection Reason

### Observation

The `decision.rejection_reason` variable contains a high proportion of missing values.

However, this missingness is **structurally driven**:

- The field is only applicable when `loan_approved = False`
- Approved loans do not have a rejection reason

Therefore, missing values in this column are not data quality errors but reflect business logic.

---

### Data Quality Interpretation

- **Consistency:** Categories are consistently formatted.
- **Completeness:** Missingness is structurally conditional.
- **Validity:** The variable behaves as expected given approval status.

No remediation was required for this field.

## 6. Invalid / Implausible Values

In [28]:
# ============================================================
# 6.1 Plausibility Check: Debt-to-Income Ratio
# ============================================================

col = "financials.debt_to_income"

print("DTI summary statistics:")
display(df[col].describe())

invalid_dti = df[(df[col] < 0) | (df[col] > 2)]

print("\nNumber of implausible DTI values:", invalid_dti.shape[0])
display(invalid_dti[["_id", col]])

DTI summary statistics:


count    500.000000
mean       0.245520
std        0.136148
min        0.050000
25%        0.150000
50%        0.230000
75%        0.342500
max        1.850000
Name: financials.debt_to_income, dtype: float64


Number of implausible DTI values: 0


,_id,financials.debt_to_income


## Invalid / Implausible Values – Debt-to-Income Ratio

### Check Performed

The `financials.debt_to_income` variable was evaluated for plausibility.

Expected range:
- ≥ 0
- Typically ≤ 1
- Extreme values > 2 considered implausible

---

### Summary Statistics

- Min: 0.05  
- Max: 1.85  
- Mean: 0.25  

No negative values or extreme outliers were detected.

---

### Result

Number of implausible DTI values: **0**

---

### Data Quality Interpretation

- **Validity:** All values fall within a realistic financial range.
- **Accuracy:** No evidence of data entry errors.
- **Remediation:** Not required.

The debt-to-income variable is considered analytically reliable.

In [29]:
# ============================================================
# 6.2 Plausibility Check: Interest Rate
# ============================================================

col = "decision.interest_rate"

print("Interest rate summary statistics:")
display(df[col].describe())

# Define implausible thresholds
invalid_rate = df[(df[col] < 0) | (df[col] > 100)]

print("\nNumber of implausible interest rates:", invalid_rate.shape[0])
display(invalid_rate[["_id", col]])

Interest rate summary statistics:


count    292.000000
mean       4.564726
std        1.162866
min        2.500000
25%        3.500000
50%        4.550000
75%        5.600000
max        6.500000
Name: decision.interest_rate, dtype: float64


Number of implausible interest rates: 0


,_id,decision.interest_rate


## Invalid / Implausible Values – Interest Rate

### Check Performed

The `decision.interest_rate` variable was evaluated for plausibility.

Expected range:
- ≥ 0%
- Realistically below extreme thresholds (e.g., < 100%)

Note: This field is only populated for approved loans.

---

### Summary Statistics

- Min: 2.5%  
- Max: 6.5%  
- Mean: 4.56%  

All values fall within a realistic lending range.

---

### Result

Number of implausible interest rate values: **0**

---

### Data Quality Interpretation

- **Validity:** All values are within expected financial bounds.
- **Accuracy:** No extreme or erroneous interest rates detected.
- **Remediation:** Not required.

The interest rate variable is considered analytically sound.

In [31]:
# ============================================================
# 6.3 Plausibility Check: Savings Balance
# ============================================================

col = "financials.savings_balance"

print("Savings balance summary statistics:")
display(df[col].describe())

# Check for negative values
negative_savings = df[df[col] < 0]

print("\nNumber of negative savings values:", negative_savings.shape[0])
display(negative_savings[["_id", col]])

Savings balance summary statistics:


count      500.000000
mean     29579.530000
std      16745.805308
min      -5000.000000
25%      17387.250000
50%      27399.000000
75%      38281.750000
max      88078.000000
Name: financials.savings_balance, dtype: float64


Number of negative savings values: 1


,_id,financials.savings_balance
159,app_290,-5000


In [32]:
# ============================================================
# Remediation: Negative savings balance
# ============================================================

col = "financials.savings_balance"

df.loc[df[col] < 0, col] = np.nan

print("Negative savings after remediation:",
      (df[col] < 0).sum())

print("Missing savings after remediation:",
      df[col].isna().sum())

Negative savings after remediation: 0
Missing savings after remediation: 1


## Invalid / Implausible Values – Savings Balance

### Check Performed

The `financials.savings_balance` variable was evaluated for plausibility.

Expected range:
- ≥ 0
- Realistic upper values (no extreme anomalies detected)

---

### Findings

- 1 record contained a negative value (-5000)
- All other values fell within a realistic range

A negative savings balance is inconsistent with the semantic meaning of the variable.

---

### Remediation

The negative value was set to `NaN`, as the correct value could not be reliably inferred.

---

### Data Quality Interpretation

- **Validity:** One implausible value detected.
- **Accuracy:** Corrected via controlled nullification.
- **Remediation:** Minimal impact (1/500 records).

The savings balance variable is now considered valid for analysis.

## Final Structural Cleanup

The original income-related columns have been removed to prevent ambiguity.

The dataset now contains:
- 500 unique applications
- 20 structured variables
- A single consolidated income field

This ensures schema clarity and reduces the risk of accidental misuse in downstream analysis.

In [15]:
df_clean["applicant_info.gender"].value_counts(dropna=False)

NameError: name 'df_clean' is not defined

## Categorical Consistency Issue – Gender

The `applicant_info.gender` field contains inconsistent encodings:

- Full labels: "Male", "Female"
- Abbreviations: "M", "F"
- A small number of irregular or blank values

Such inconsistencies may distort fairness metrics if categories are treated separately.

Plan:
- Standardize values to lowercase
- Map abbreviations ("m" → "male", "f" → "female")
- Convert blank or invalid entries to null

In [ ]:
df_clean["applicant_info.gender"] = (
    df_clean["applicant_info.gender"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({"m": "male", "f": "female"})
)

df_clean["applicant_info.gender"].value_counts(dropna=False)

applicant_info.gender
female    251
male      247
            2
Name: count, dtype: int64

## Gender Standardization Result

The gender field has been normalized to a consistent lowercase format:

- "male"
- "female"

Abbreviations and casing inconsistencies have been resolved.

Two records remain with invalid or blank gender values. 
These will be treated as missing rather than forcefully assigned, to preserve analytical integrity.

In [ ]:
df_clean["applicant_info.gender"] = df_clean["applicant_info.gender"].replace(
    ["", "nan", "none"], np.nan
)

df_clean["applicant_info.gender"].isna().sum()

np.int64(2)

## Final Gender Cleanup

Invalid or blank gender values have been converted to null (NaN).

This preserves data integrity without artificially assigning categories.

The dataset now contains:
- 251 female
- 247 male
- 2 missing

The field is now consistent and ready for fairness analysis.

In [ ]:
df_clean["applicant_info.date_of_birth"].head(10)

383    1986-05-27
339    1999-08-01
284    1982-08-24
255    02/28/1995
136    1960/06/19
328    1987-07-12
12     1989-06-13
81     1993-07-12
390    1989-11-20
47     1996-09-07
Name: applicant_info.date_of_birth, dtype: object

## Date Format Inconsistency – Date of Birth

The `applicant_info.date_of_birth` field is stored as a string and contains multiple date formats:

- ISO format: `YYYY-MM-DD`
- Slash-separated format: `YYYY/MM/DD`
- US-style format: `MM/DD/YYYY`

Such inconsistencies prevent reliable parsing and can lead to incorrect age calculations.

Plan: parse the field into a proper datetime type using robust parsing rules.

In [ ]:
dob_raw = (
    df_clean["applicant_info.date_of_birth"]
    .astype(str)
    .str.strip()
    .replace({"": np.nan, "nan": np.nan, "none": np.nan})
)

# Normalize separators: "/" -> "-"
dob_norm = dob_raw.str.replace("/", "-", regex=False)

# Try YYYY-MM-DD first
dob_parsed = pd.to_datetime(dob_norm, errors="coerce", format="%Y-%m-%d")

# For remaining, try DD-MM-YYYY
mask = dob_parsed.isna()
dob_parsed.loc[mask] = pd.to_datetime(dob_norm[mask], errors="coerce", format="%d-%m-%Y")

df_clean["applicant_info.date_of_birth_parsed"] = dob_parsed

df_clean["applicant_info.date_of_birth_parsed"].isna().sum()

np.int64(30)

In [ ]:
mask_final = df_clean["applicant_info.date_of_birth_parsed"].isna()
mask_final.sum(), df_clean.loc[mask_final, "applicant_info.date_of_birth"].head(20)

(np.int64(30),
 255    02/28/1995
 378    08/26/1983
 441    07/15/1999
 420    03/16/1970
 26               
 275              
 451    05/21/1984
 469    11/26/1975
 348    10/24/2001
 88     12/16/1985
 462              
 313    04/13/1996
 330    09/20/1998
 498    11/15/1985
 327    06/20/2000
 345    12/13/1970
 183    06/25/1994
 226    04/25/1997
 115    12/28/1972
 134    07/28/1988
 Name: applicant_info.date_of_birth, dtype: object)

In [ ]:
mask_us = dob_parsed.isna()
dob_parsed.loc[mask_us] = pd.to_datetime(dob_norm[mask_us], errors="coerce", format="%m-%d-%Y")

df_clean["applicant_info.date_of_birth_parsed"] = dob_parsed
df_clean["applicant_info.date_of_birth_parsed"].isna().sum()

np.int64(4)

In [ ]:
mask_missing_dob = df_clean["applicant_info.date_of_birth_parsed"].isna()

df_clean.loc[
    mask_missing_dob,
    ["_id", "applicant_info.date_of_birth", "applicant_info.date_of_birth_parsed"]
]

,_id,applicant_info.date_of_birth,applicant_info.date_of_birth_parsed
26,app_075,,NaT
275,app_120,,NaT
462,app_165,,NaT
448,app_350,,NaT


## Date of Birth Cleaning Summary

The `applicant_info.date_of_birth` column contained multiple date formats, including:

- `YYYY-MM-DD`
- `DD/MM/YYYY`
- `MM/DD/YYYY`
- Empty strings

### Cleaning Steps

1. Empty strings were converted to `NaN`.
2. All separators (`/`) were normalized to `-`.
3. Dates were parsed using multiple format attempts:
   - `YYYY-MM-DD`
   - `DD-MM-YYYY`
   - `MM-DD-YYYY`

### Result

- 496 records were successfully parsed into proper datetime format.
- 4 records contain genuinely missing date values.
- These missing values are represented as `NaT`.

### Data Quality Impact

This process ensures:

- Consistent datetime formatting across the dataset  
- No silent parsing failures  
- Transparent handling of missing values  
- Improved reliability for downstream feature engineering  

The dataset is now ready for age feature engineering and GDPR-compliant transformation steps.

In [ ]:
today = pd.Timestamp.today()

df_clean["applicant_age"] = (
    (today - df_clean["applicant_info.date_of_birth_parsed"])
    .dt.days // 365
)

df_clean["applicant_age"].describe()

count    496.000000
mean      40.747984
std       10.964912
min       23.000000
25%       32.000000
50%       39.000000
75%       47.000000
max       67.000000
Name: applicant_age, dtype: float64

## GDPR and Privacy Compliance Step

The dataset currently contains personally identifiable information (PII), including:

- Full name  
- Email address  
- Social security number (SSN)  
- IP address  
- Date of birth  

While some of these fields were necessary for data validation and feature engineering (e.g., calculating age), they are not required for downstream modeling.

To ensure GDPR compliance and privacy protection:

- All direct identifiers will be removed.
- Date of birth will be replaced by the derived feature `applicant_age`.
- System-generated notes will also be removed as they may contain sensitive operational information.

After this step, the dataset will contain only analytical and model-relevant features.

In [ ]:
pii_columns = [
    "applicant_info.full_name",
    "applicant_info.email",
    "applicant_info.ssn",
    "applicant_info.ip_address",
    "applicant_info.date_of_birth",
    "applicant_info.date_of_birth_parsed",
    "notes"
]

df_model_ready = df_clean.drop(columns=pii_columns)

df_model_ready.shape

(500, 15)

## GDPR Compliance Verification

All personally identifiable information (PII) has now been removed from the dataset.

Removed columns include:

- `applicant_info.full_name`
- `applicant_info.email`
- `applicant_info.ssn`
- `applicant_info.ip_address`
- `applicant_info.date_of_birth`
- `applicant_info.date_of_birth_parsed`
- `notes`

The dataset now contains only analytical features relevant for modeling.

Final dataset shape: **500 rows × 15 columns**

This ensures GDPR compliance and prevents exposure of sensitive personal data in downstream analytics or machine learning workflows.

In [ ]:
df_model_ready.info()

<class 'pandas.core.frame.DataFrame'>
Index: 500 entries, 383 to 152
Data columns (total 15 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   _id                               500 non-null    object 
 1   spending_behavior                 500 non-null    object 
 2   processing_timestamp              62 non-null     object 
 3   applicant_info.gender             498 non-null    object 
 4   applicant_info.zip_code           500 non-null    object 
 5   financials.credit_history_months  500 non-null    int64  
 6   financials.debt_to_income         500 non-null    float64
 7   financials.savings_balance        500 non-null    int64  
 8   decision.loan_approved            500 non-null    bool   
 9   decision.rejection_reason         208 non-null    object 
 10  loan_purpose                      50 non-null     object 
 11  decision.interest_rate            292 non-null    float64
 12  decision.ap

In [ ]:
df_model_ready.isna().sum().sort_values(ascending=False)

loan_purpose                        450
processing_timestamp                438
decision.rejection_reason           292
decision.interest_rate              208
decision.approved_amount            208
applicant_age                         4
applicant_info.gender                 2
_id                                   0
spending_behavior                     0
applicant_info.zip_code               0
financials.credit_history_months      0
financials.debt_to_income             0
financials.savings_balance            0
decision.loan_approved                0
financials.final_income               0
dtype: int64

## Data Quality Assessment – Completeness Analysis

Total records: **500**

### 1. loan_purpose
- Missing: 450 / 500 (90%)
- This represents extremely low completeness.
- The field may be relevant for proxy discrimination analysis.
- Indicates weak input validation or lack of mandatory field enforcement.

**Assessment:** Major data quality issue (Completeness dimension).

---

### 2. processing_timestamp
- Missing: 438 / 500 (87.6%)
- Represents a significant governance gap.

From a governance perspective:
- Weak audit trail
- Limited traceability of decision timing
- Potential non-compliance with GDPR accountability principle
- Potential transparency issues under the EU AI Act

**Assessment:** Critical governance and auditability issue.

---

### 3. decision.rejection_reason
- Missing: 292 records
- Likely corresponds to approved loans (logical absence).

This is not necessarily a completeness issue but requires a consistency check to ensure:
- Rejected loans always have a rejection reason.
- Approved loans do not incorrectly contain rejection reasons.

**Assessment:** Requires consistency validation.

---

### 4. decision.interest_rate and decision.approved_amount
- Missing: 208 records each
- Likely corresponds to rejected loans.

This is structurally expected but must be validated for logical consistency.

**Assessment:** Not inherently a data quality issue; requires validation.

---

### 5. applicant_age
- Missing: 4 (0.8%)
- Acceptable level of missingness.

---

### 6. applicant_info.gender
- Missing: 2 (0.4%)
- Low missingness but relevant for bias analysis.

---

### Summary

The most significant data quality issues relate to:

- Severe incompleteness of `loan_purpose`
- Critical governance gap in `processing_timestamp`

Other missing values appear structurally linked to decision outcomes and require consistency validation rather than remediation.

In [ ]:
# Check logical consistency
pd.crosstab(
    df_model_ready["decision.loan_approved"],
    df_model_ready["decision.rejection_reason"].isna()
)

decision.rejection_reason,False,True
decision.loan_approved,,
False,208,0
True,0,292


## Logical Consistency Validation – Decision Fields

A cross-tabulation was performed between:

- `decision.loan_approved`
- Presence of `decision.rejection_reason`

### Results:

- All rejected applications (208) contain a rejection reason.
- No approved applications contain a rejection reason.
- No inconsistent cases were detected.

### Interpretation

The dataset demonstrates perfect logical consistency between loan approval status and rejection reason.

This confirms structural integrity of the decision fields and indicates no internal contradictions in outcome recording.

**Data Quality Dimension:** Consistency  
**Result:** No remediation required

In [ ]:
pd.crosstab(
    df_model_ready["decision.loan_approved"],
    df_model_ready["decision.interest_rate"].isna()
)


decision.interest_rate,False,True
decision.loan_approved,,
False,0,208
True,292,0


## Logical Consistency Validation – Interest Rate

A cross-tabulation was performed between:

- `decision.loan_approved`
- Presence of `decision.interest_rate`

### Results:

- All rejected applications (208) have no interest rate.
- All approved applications (292) have a valid interest rate.
- No inconsistencies were detected.

### Interpretation

Interest rates are correctly assigned only to approved loans.

The relationship between loan approval and interest rate is logically consistent across the dataset.

**Data Quality Dimension:** Consistency  
**Result:** No remediation required

In [ ]:
pd.crosstab(
    df_model_ready["decision.loan_approved"],
    df_model_ready["decision.approved_amount"].isna()
)


decision.approved_amount,False,True
decision.loan_approved,,
False,0,208
True,292,0


## Logical Consistency Validation – Approved Amount

A cross-tabulation was performed between:

- `decision.loan_approved`
- Presence of `decision.approved_amount`

### Results:

- All rejected applications (208) have no approved amount.
- All approved applications (292) contain a valid approved amount.
- No inconsistencies were detected.

### Interpretation

Approved amounts are correctly assigned only to approved loans.

The dataset demonstrates strong logical consistency between approval status and loan amount allocation.

**Data Quality Dimension:** Consistency  
**Result:** No remediation required

## Plausibility Checks (Outlier Validation)

### Debt-to-Income Ratio

In [ ]:
df_model_ready["financials.debt_to_income"].describe()

count    500.000000
mean       0.245520
std        0.136148
min        0.050000
25%        0.150000
50%        0.230000
75%        0.342500
max        1.850000
Name: financials.debt_to_income, dtype: float64

In [ ]:
df_model_ready[df_model_ready["financials.debt_to_income"] < 0]

,_id,spending_behavior,processing_timestamp,applicant_info.gender,applicant_info.zip_code,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.final_income,applicant_age


In [ ]:
df_model_ready[df_model_ready["financials.debt_to_income"] > 1]

,_id,spending_behavior,processing_timestamp,applicant_info.gender,applicant_info.zip_code,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.final_income,applicant_age
316,app_402,"[{'category': 'Education', 'amount': 482}]",NaN,female,90214,27,1.85,37281,True,NaN,NaN,3.2,17000.0,88000.0,61.0


### Savings Balance

In [ ]:
df_model_ready["financials.savings_balance"].describe()

count      500.000000
mean     29579.530000
std      16745.805308
min      -5000.000000
25%      17387.250000
50%      27399.000000
75%      38281.750000
max      88078.000000
Name: financials.savings_balance, dtype: float64

In [ ]:
df_model_ready[df_model_ready["financials.savings_balance"] < 0]

,_id,spending_behavior,processing_timestamp,applicant_info.gender,applicant_info.zip_code,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.final_income,applicant_age
159,app_290,"[{'category': 'Fitness', 'amount': 599}]",NaN,female,90221,39,0.27,-5000,True,NaN,NaN,4.3,49000.0,104000.0,36.0


### Interest Rate

In [ ]:
df_model_ready["decision.interest_rate"].describe()

count    292.000000
mean       4.564726
std        1.162866
min        2.500000
25%        3.500000
50%        4.550000
75%        5.600000
max        6.500000
Name: decision.interest_rate, dtype: float64

In [ ]:
df_model_ready[df_model_ready["decision.interest_rate"] < 0]

,_id,spending_behavior,processing_timestamp,applicant_info.gender,applicant_info.zip_code,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.final_income,applicant_age


### Applicant Age

In [ ]:
df_model_ready["applicant_age"].describe()

count    496.000000
mean      40.747984
std       10.964912
min       23.000000
25%       32.000000
50%       39.000000
75%       47.000000
max       67.000000
Name: applicant_age, dtype: float64

In [ ]:
df_model_ready[
    (df_model_ready["applicant_age"] < 18) |
    (df_model_ready["applicant_age"] > 100)
]

,_id,spending_behavior,processing_timestamp,applicant_info.gender,applicant_info.zip_code,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.final_income,applicant_age


## Plausibility Check – Results

The following observations were made:

### Debt-to-Income Ratio
Values ranged from 0.05 to 1.85.
Ratios above 1 indicate high financial leverage but are economically plausible.
No invalid values were detected.

### Savings Balance
Negative balances (minimum: -5000) were observed.
These may reflect overdraft or temporary negative liquidity and are considered realistic.

### Interest Rate
Interest rates ranged between 2.5 and 6.5.
No negative or extreme values were detected.

### Applicant Age
Applicant age ranged from 23 to 67.
No unrealistic ages (<18 or >100) were found.

### Conclusion

All numerical variables fall within realistic and economically plausible ranges.
No corrective data remediation was required.